In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
# import sk learn libraries
import sklearn.tree
import sklearn.linear_model
import sklearn.metrics
import sklearn.ensemble
from sklearn.model_selection import cross_val_score, KFold, GroupKFold, StratifiedKFold

import cv2

## Load Data

In [ ]:
img_file = "x_train_img.npz"
metadata_file = "x_train.csv"
y_file = "y_train.csv"

In [ ]:
X_dev = pd.read_csv(metadata_file)
y_dev = pd.read_csv(y_file)
X_dev

In [ ]:
y_dev.head()

In [ ]:
# Function and code to load images
def load_img_data(file_path):
    with np.load(file_path) as data:
        img = data['images']
        ids = data['image_ids']
    print(f"Successfully loaded {img.shape[0]} images.")  
    return img, ids

imgs, img_ids = load_img_data(img_file)
print(img_ids.shape, imgs.shape)

# Make sure the IDs match and are in the same order
assert np.all(img_ids == X_dev['img_id']), "Image IDs in metadata and image data do not match"

In [ ]:
# Plot the class balance
sns.countplot(y_dev["fine_label"])

In [ ]:
# Show the first 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 5))
for i in range(5):
    axes[i].imshow(imgs[i])
    axes[i].axis('off')
    axes[i].set_title(f"Sample {i} ({y_dev.iloc[i]["fine_label"]})")

plt.show()

## Encode Categorical Features

We'll convert all categorical features to one-hot encodings, and all binary features to 1/0. 
We can auto-detect the binary features by looking for "True" and "False" (after manually looking through the data to confirm that's how these features are encoded.)

There also is a column labeled "gender". In publicly available datasets it can be especially tricky to figure out what a gender or sex column actually represents. Sometimes it's a patient's legal sex from government records, sometimes it's their biological sex from medical records, sometimes it's the clinician's assumption of the patient's gender, and sometimes it's the patient's self identified gender. The authors of this dataset provide a pdf of [the translated data collection instrument](https://pmc.ncbi.nlm.nih.gov/articles/instance/7479321/bin/mmc2.pdf) where we can see that gender is listed under "Questions About the Patient" and are phrased as questions the clinician would ask the patient directly. Therefore, in this case the gender column most likely represents patient self-identified gender. 

In [ ]:
# These are the 3 columns in X that have categorical data.
# Everything else is binary or numeric. 
categorical_cols = ['background_father', 'background_mother', 'region'] 

# Get all obvious true/false cols that aren't already true/false
for col in X_dev.select_dtypes(include=['object']):
    if X_dev[col].dropna().astype(str).str.contains('True|False').any():
        print(f"Automatically converting {col} to binary...")
        unmapped = X_dev[~X_dev[col].isin(['True','False'])][col].unique()
        print("The following values are being mapped to NaN in this column: ", unmapped)
        X_dev[col] = X_dev[col].map({'True': 1, 'False': 0, True: 1, False: 0})

X_dev['gender'] = X_dev['gender'].map({'MALE': 0, 'FEMALE': 1})

# Handle the categorical data
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

encoded_feats = encoder.fit_transform(X_dev[categorical_cols])
encoded_df = pd.DataFrame(
    encoded_feats, 
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_dev.index
)

# Drop original text columns and join the new numeric ones
X_dev = X_dev.drop(columns=categorical_cols).join(encoded_df)


### Make age groups feature

In [ ]:
age_groups = pd.cut(
    X_dev['age'], 
    bins=[0, 30, 60, 100], 
    labels=['Young (<30)', 'Adult (30-60)', 'Senior (60+)']
)

## Handle Missing Data

In [ ]:
# First we'll calculate how much missing data there is
X_dev.isna().sum()

In [ ]:
# Then we'll decide what to do. For now we'll just treat missing data as false, 
# since it probably means that a patient didn't think the question was important or relevant.
X_dev = X_dev.fillna(0)
X_dev

## Your Code

You might want to revisit the above preprocessing steps for your Problem 2 model. You also might want to put some of the above code into functions to make it easier to apply to different datasets. 

In [ ]:
def calculate_region_contrast(img_rgb):
    """
    Calculates the mean intensity difference between the lesion 
    and the surrounding skin.
    """
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    kernel = np.ones((5, 5), np.uint8)
    dilated_mask = cv2.dilate(mask, kernel, iterations=4)
    outer_mask = cv2.subtract(dilated_mask, mask)

    inner_pixels = gray[mask > 0]
    outer_pixels = gray[outer_mask > 0]

    # Error Case
    if inner_pixels.size == 0 or outer_pixels.size == 0:
        return 0.0
    
    contrast = np.abs(np.mean(inner_pixels) - np.mean(outer_pixels))

    return contrast


In [ ]:
# print out the column names
X_dev.columns

Add our two features created from the image data:

In [ ]:
# Add image contrast as a feature to the dataset
image_contrast_values = [calculate_region_contrast(img) for img in imgs]
X_dev['img_contrast'] = image_contrast_values

In [ ]:
# Add color variation feature

color_N_3 = np.std(imgs, axis=(1, 2))
R = color_N_3[:, 0]
G = color_N_3[:, 1]
B = color_N_3[:, 2]

X_dev['red_variation'] = R
X_dev['green_variation'] = G
X_dev['blue_variation'] = B

In [ ]:
X_dev.head()

In [ ]:
# create a list of features to use
feature_list = ['smoke', 'drink', 'age', 'pesticide',
                'gender', 'skin_cancer_history', 'cancer_history'
                ,'has_piped_water', 'has_sewage_system', 'fitspatrick',

                'region_HAND', 'region_LIP', 'region_NECK', 'region_NOSE', 'region_SCALP',
                'region_THIGH', 'region_ABDOMEN', 'region_ARM', 'region_BACK', 'region_EAR',
                'region_FACE', 'region_FOOT', 'region_FOREARM', 'region_CHEST',

                'diameter_1', 'diameter_2', 'itch', 'grew',
                'hurt', 'changed', 'bleed', 'elevation', 'img_contrast',
                'red_variation', 'green_variation', 'blue_variation']

In [ ]:
# Base Forest
base_forest = sklearn.ensemble.RandomForestClassifier(criterion='entropy')

In [ ]:
# Hyperparameter Grid
forest_hyperparameter_grid = dict(
    max_features=[3, 10, 33, 100, 333],
    max_depth=[4,8,16,32],
    min_samples_leaf=[1,5,10],
    n_estimators=[100,500,1000]
    )

In [ ]:
# Set up the cross validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# Set up the actual model selection using gridsearch
forest_searcher = sklearn.model_selection.GridSearchCV(
    estimator = base_forest,
    param_grid = forest_hyperparameter_grid,
    scoring = 'roc_auc',
    cv = kf,
    return_train_score = True,
    refit = True,
    n_jobs = -1)

In [ ]:
# Create the datasers to fit
x_train = X_dev[feature_list]
y_train = y_dev['coarse_label']

In [ ]:
# Fit the model
forest_searcher.fit(x_train,y_train)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results = pd.DataFrame(forest_searcher.cv_results_)

# Filter to a fixed set of other params to isolate the effect of max_depth
filtered = results[
    (results['param_n_estimators'] == 100) &
    (results['param_min_samples_leaf'] == 1) &
    (results['param_max_features'] == 33)
]

# Group by max_depth and average scores
grouped = filtered.groupby('param_max_depth').agg(
    mean_train=('mean_train_score', 'mean'),
    mean_val=('mean_test_score', 'mean')
).reset_index()

plt.plot(grouped['param_max_depth'], grouped['mean_train'], label='Train AUC', marker='o')
plt.plot(grouped['param_max_depth'], grouped['mean_val'], label='Validation AUC', marker='o')
plt.xlabel('max_depth')
plt.ylabel('ROC AUC')
plt.title('Train vs Validation AUC by max_depth')
plt.legend()
plt.show()

In [ ]:
# load the test data
x_test = pd.read_csv('x_test.csv')
img_file_test = 'x_test_img.npz'
test_imgs, test_img_ids = load_img_data(img_file_test)

In [ ]:
# make sure the image ids match
assert np.all(test_img_ids == x_test['img_id']), "Image IDs in metadata and image data do not match"

In [ ]:
# Encode the test categorical data
# Everything else is binary or numeric. 
categorical_cols = ['background_father', 'background_mother', 'region'] 

# Get all obvious true/false cols that aren't already true/false
for col in x_test.select_dtypes(include=['object']):
    if x_test[col].dropna().astype(str).str.contains('True|False').any():
        print(f"Automatically converting {col} to binary...")
        unmapped = x_test[~x_test[col].isin(['True','False'])][col].unique()
        print("The following values are being mapped to NaN in this column: ", unmapped)
        x_test[col] = x_test[col].map({'True': 1, 'False': 0, True: 1, False: 0})

x_test['gender'] = x_test['gender'].map({'MALE': 0, 'FEMALE': 1})

# Handle the categorical data
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

encoded_feats = encoder.fit_transform(x_test[categorical_cols])
encoded_df = pd.DataFrame(
    encoded_feats, 
    columns=encoder.get_feature_names_out(categorical_cols),
    index=x_test.index
)

# Drop original text columns and join the new numeric ones
x_test = x_test.drop(columns=categorical_cols).join(encoded_df)


In [ ]:
# make age groups feature
age_groups = pd.cut(
    x_test['age'], 
    bins=[0, 30, 60, 100], 
    labels=['Young (<30)', 'Adult (30-60)', 'Senior (60+)']
)

In [ ]:
# get rid of missing test data
x_test = x_test.fillna(0)
x_test

In [ ]:
# Add image contrast as a feature to the test dataset
image_contrast_values = [calculate_region_contrast(test_img) for test_img in test_imgs]
x_test['img_contrast'] = image_contrast_values

In [ ]:
# Add color variation feature

color_test = np.std(test_imgs, axis=(1, 2))
R_test = color_test[:, 0]
G_test = color_test[:, 1]
B_test = color_test[:, 2]

x_test['red_variation'] = R_test
x_test['green_variation'] = G_test
x_test['blue_variation'] = B_test

In [ ]:
#Get the correct columns
x_test_final = x_test[feature_list]

In [ ]:
# get the final predictions
y_proba = forest_searcher.predict_proba(x_test_final)[:,1]
np.savetxt('yproba1_test.txt', y_proba)
